# CHAPTER 6 : Data Loading, Storage, and File Formats

----

In [65]:
%run "Ch00 - Basic Imports and Set ups.ipynb"

Accessing data is a necessary first step for using most of the tools in this book. I’m
going to be focused on data input and output using pandas, though there are numerous
tools in other libraries to help with reading and writing data in various formats.

Input and output typically falls into a few main categories: reading text files and other
more efficient on-disk formats, loading data from databases, and interacting with network
sources like web APIs.

---

## Basic Imports and Set ups

In [27]:
import pandas as pd
import numpy as np
import os
from faker import Faker

This is a function to render dataframes as tables in the Jupyter Notebook.

In [28]:
import pandas as pd
from IPython.display import display, HTML

def render_book_table(title, columns, rows):
    """
    Render a documentation-style table in Jupyter.

    Parameters
    ----------
    title : str
        Title displayed above the table
    columns : list
        Column names
    rows : list of lists
        Table data rows
    """

    table_data = pd.DataFrame(rows, columns=columns)

    styled_table = (
        table_data.style
        .hide(axis="index")
        .set_table_styles([
            {"selector": "th",
             "props": [("background-color", "#f2f2f2"),
                       ("font-weight", "bold"),
                       ("text-align", "center"),
                       ("border", "1px solid #999"),
                       ("padding", "8px")]},

            {"selector": "td",
             "props": [("border", "1px solid #999"),
                       ("padding", "8px"),
                       ("white-space", "normal"),
                       ("text-align", "left")]},

            {"selector": "table",
             "props": [("border-collapse", "collapse"),
                       ("width", "100%")]}
        ])
    )

    display(HTML(f"<h3>{title}</h3>"))
    display(styled_table)

Install Faker library in the same Pthon environment which Jupyter uses for itself. Faker is useful for making meaningful fake files, dataframes etc...

`pip install faker`

## 6.1 Reading and Writing Data in Text Format
-----

pandas features a number of functions for reading tabular data as a DataFrame
object. Table 6-1 summarizes some of them, though `read_csv` and `read_table` are
likely the ones you’ll use the most.

In [29]:
columns = ["Function", "Description"]

rows = [
["read_csv","Load delimited data from a file, URL, or file-like object; use comma as default delimiter"],
["read_table","Load delimited data from a file, URL, or file-like object; use tab ('\\t') as default delimiter"],
["read_fwf","Read data in fixed-width column format (i.e., no delimiters)"],
["read_clipboard","Version of read_table that reads data from the clipboard"],
["read_excel","Read tabular data from an Excel XLS or XLSX file"],
["read_hdf","Read HDF5 files written by pandas"],
["read_html","Read all tables found in the given HTML document"],
["read_json","Read data from a JSON string representation"],
["read_msgpack","Read pandas data encoded using the MessagePack binary format"],
["read_pickle","Read an arbitrary object stored in Python pickle format"],
["read_sas","Read a SAS dataset stored in SAS custom storage format"],
["read_sql","Read the results of a SQL query as a pandas DataFrame"],
["read_stata","Read a dataset from Stata file format"],
["read_feather","Read the Feather binary file format"]
]

render_book_table(
    "Table 6-1. Parsing Functions in pandas",
    columns,
    rows
)

Function,Description
read_csv,"Load delimited data from a file, URL, or file-like object; use comma as default delimiter"
read_table,"Load delimited data from a file, URL, or file-like object; use tab ('\t') as default delimiter"
read_fwf,"Read data in fixed-width column format (i.e., no delimiters)"
read_clipboard,Version of read_table that reads data from the clipboard
read_excel,Read tabular data from an Excel XLS or XLSX file
read_hdf,Read HDF5 files written by pandas
read_html,Read all tables found in the given HTML document
read_json,Read data from a JSON string representation
read_msgpack,Read pandas data encoded using the MessagePack binary format
read_pickle,Read an arbitrary object stored in Python pickle format


----
I’ll give an overview of the mechanics of these functions, which are meant to convert
text data into a DataFrame. The optional arguments for these functions may fall into
a few categories:

***Indexing***
> Can treat one or more columns as the returned DataFrame, and whether to get
column names from the file, the user, or not at all.

***Type inference and data conversion***
> This includes the user-defined value conversions and custom list of missing value
markers.

***Datetime parsing***
> Includes combining capability, including combining date and time information
spread over multiple columns into a single column in the result.

***Iterating*** 
> Support for iterating over chunks of very large files.

Because of how messy data in the real world can be, some of the data loading functions
(especially `read_csv`) have grown very complex in their options over time. It’s
normal to feel overwhelmed by the number of different parameters (`read_csv` has
over 50 as of this writing). The online pandas documentation has many examples
about how each of them works, so if you’re struggling to read a particular file, there
might be a similar enough example to help you find the right parameters.
Some of these functions, like pandas.read_csv, perform type inference, because the
column data types are not part of the data format. That means you don’t necessarily
have to specify which columns are numeric, integer, boolean, or string. Other data
formats, like HDF5, Feather, and msgpack, have the data types stored in the format.

---

Handling dates and other custom types can require extra effort. Let’s start with a
small comma-separated (CSV) text file:m

In [30]:
!mkdir examples

A subdirectory or file examples already exists.


In [31]:
%%writefile examples/ex1.csv
a,b,c,d,message
1,2,3,4,hello
5,6,7,8,world
9,10,11,12,foo

Overwriting examples/ex1.csv


Since this is comma-delimited, we can use `read_csv` to read it into a DataFrame:

In [32]:
df = pd.read_csv('examples/ex1.csv')

In [33]:
df

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


In [34]:
pd.read_table('examples/ex1.csv', sep=',')

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


A file will not always have a header row. Consider this file:

In [35]:
%%writefile examples/ex2.csv
1,2,3,4,hello
5,6,7,8,world
9,10,11,12,foo

Overwriting examples/ex2.csv


To read this file, you have a couple of options. You can allow pandas to assign default
column names, or you can specify names yourself:

In [36]:
pd.read_csv('examples/ex2.csv', header=None)

,0,1,2,3,4
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


In [37]:
pd.read_csv('examples/ex2.csv', names=['a', 'b', 'c', 'd', 'message'])

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


---

Suppose you wanted the `message` column to be the index of the returned DataFrame.
You can either indicate you want the column at index 4 or named 'message' using
the `index_col` argument:

In [38]:
names = ['a', 'b', 'c', 'd', 'message']

In [39]:
pd.read_csv('examples/ex2.csv', names=names, index_col='message')

,a,b,c,d
message,,,,
hello,1,2,3,4
world,5,6,7,8
foo,9,10,11,12


-----

In the event that you want to form a hierarchical index from multiple columns, pass a
list of column numbers or names:

In [40]:
%%writefile examples/csv_mindex.csv
key1,key2,value1,value2
one,a,1,2
one,b,3,4
one,c,5,6
one,d,7,8
two,a,9,10
two,b,11,12
two,c,13,14
two,d,15,16

Overwriting examples/csv_mindex.csv


In [41]:
parsed = pd.read_csv('examples/csv_mindex.csv',index_col=['key1', 'key2'])

In [42]:
parsed

value1  value2
key1 key2                
one  a          1       2
     b          3       4
     c          5       6
     d          7       8
two  a          9      10
     b         11      12
     c         13      14
     d         15      16

In some cases, a table might not have a fixed delimiter, using whitespace or some
other pattern to separate fields. Consider a text file that looks like this:

In [43]:
%%writefile examples/ex3.txt
            A         B         C
aaa -0.264438 -1.026059 -0.619500
bbb 0.927272 0.302904 -0.032399
ccc -0.264273 -0.386314 -0.217601
ddd -0.871858 -0.348382 1.100491

Overwriting examples/ex3.txt


In [44]:
result = pd.read_table('examples/ex3.txt', sep='\s+')

In [45]:
result

,A,B,C
aaa,-0.264438,-1.026059,-0.619500
bbb,0.927272,0.302904,-0.032399
ccc,-0.264273,-0.386314,-0.217601
ddd,-0.871858,-0.348382,1.100491


Because there was one fewer column name than the number of data rows,
`read_table` infers that the first column should be the DataFrame’s index in this special
case.

-----

The parser functions have many additional arguments to help you handle the wide
variety of exception file formats that occur (see a partial listing in Table 6-2). For
example, you can skip the first, third, and fourth rows of a file with skiprows:

In [46]:
%%writefile examples/ex4.csv
# hey!
a,b,c,d,message
# just wanted to make things more difficult for you
# who reads CSV files with computers, anyway?
1,2,3,4,hello
5,6,7,8,world
9,10,11,12,foo

Overwriting examples/ex4.csv


In [47]:
pd.read_csv('examples/ex4.csv', skiprows=[0, 2, 3])

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


Handling missing values is an important and frequently nuanced part of the file parsing
process. Missing data is usually either not present (empty string) or marked by
some *sentinel* value. By default, pandas uses a set of commonly occurring sentinels,
such as NA and NULL:

In [48]:
%%writefile examples/ex5.csv
something,a,b,c,d,message
one,1,2,3,4,NA
two,5,6,,8,world
three,9,10,11,12,foo

Overwriting examples/ex5.csv


In [49]:
result = pd.read_csv('examples/ex5.csv')

In [50]:
result

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


In [51]:
pd.isnull(result)

,something,a,b,c,d,message
0,False,False,False,False,False,True
1,False,False,False,True,False,False
2,False,False,False,False,False,False


The `na_values` option can take either a list or set of strings to consider missing
values:

In [52]:
result = pd.read_csv('examples/ex5.csv', na_values=['NULL'])

In [53]:
result

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


Different NA sentinels can be specified for each column in a dict:

In [54]:
sentinels = {'message': ['foo', 'NA'], 'something': ['two']}

In [55]:
pd.read_csv('examples/ex5.csv', na_values=sentinels)

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,NaN,5,6,NaN,8,world
2,three,9,10,11.0,12,NaN


Table 6-2 lists some frequently used options in `pandas.read_csv` and `pan
das.read_table`.

In [56]:
columns = ["Argument", "Description"]

rows = [
["path","String indicating filesystem location, URL, or file-like object"],
["sep or delimiter","Character sequence or regular expression to use to split fields in each row"],
["header","Row number to use as column names; defaults to 0 (first row), but should be None if there is no header row"],
["index_col","Column numbers or names to use as the row index in the result; can be a single name/number or a list of them for a hierarchical index"],
["names","List of column names for result, combine with header=None"],
["skiprows","Number of rows at beginning of file to ignore or list of row numbers (starting from 0) to skip"],
["na_values","Sequence of values to replace with NA"],
["comment","Character(s) to split comments off the end of lines"],
["parse_dates","Attempt to parse data to datetime; False by default. If True, attempt to parse all columns. Otherwise specify a list of column numbers or names to parse"],
["keep_date_col","If joining columns to parse date, keep the joined columns; False by default"],
["converters","Dict mapping column number or name to functions (e.g., {'foo': f}) applied to values in that column"],
["dayfirst","When parsing ambiguous dates treat as international format (e.g., 7/6/2012 -> June 7, 2012); False by default"],
["date_parser","Function to use to parse dates"],
["nrows","Number of rows to read from beginning of file"],
["iterator","Return a TextParser object for reading file piecewise"],
["chunksize","For iteration, size of file chunks"],
["skip_footer","Number of lines to ignore at end of file"],
["verbose","Print parser output information like number of missing values placed in non-numeric columns"],
["encoding","Text encoding for Unicode (e.g., 'utf-8')"],
["squeeze","If parsed data contains one column, return a Series"],
["thousands","Separator for thousands (e.g., ',' or '.')"]
]

render_book_table(
    "Table 6-2. Some read_csv/read_table Function Arguments",
    columns,
    rows
)

Argument,Description
path,"String indicating filesystem location, URL, or file-like object"
sep or delimiter,Character sequence or regular expression to use to split fields in each row
header,"Row number to use as column names; defaults to 0 (first row), but should be None if there is no header row"
index_col,Column numbers or names to use as the row index in the result; can be a single name/number or a list of them for a hierarchical index
names,"List of column names for result, combine with header=None"
skiprows,Number of rows at beginning of file to ignore or list of row numbers (starting from 0) to skip
na_values,Sequence of values to replace with NA
comment,Character(s) to split comments off the end of lines
parse_dates,"Attempt to parse data to datetime; False by default. If True, attempt to parse all columns. Otherwise specify a list of column numbers or names to parse"
keep_date_col,"If joining columns to parse date, keep the joined columns; False by default"


### Reading Text Files in Pieces

When processing very large files or figuring out the right set of arguments to correctly
process a large file, you may only want to read in a small piece of a file or iterate
through smaller chunks of the file.

Before we look at a large file, we make the pandas display settings more compact:

In [57]:
pd.options.display.max_rows = 10

In [58]:
result = pd.read_csv('examples/titanic.csv')

(Titanic Data set is available in https://www.kaggle.com/datasets/yasserh/titanic-dataset to download)

In [59]:
result

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


If you want to only read a small number of rows (avoiding reading the entire file),
specify that with nrows:

In [60]:
result =pd.read_csv('examples/titanic.csv', nrows=5)

In [61]:
result

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35,0,0,373450,8.0500,NaN,S


----

Use Faker to create a 1million row file. (This takes time)

In [64]:
import pandas as pd
import numpy as np
from faker import Faker

fake = Faker()

TOTAL_ROWS = 1_000_000
CHUNK_SIZE = 100_000
OUTPUT_FILE = "examples/large.csv"

departments = ['Engineering', 'Finance', 'HR', 'Marketing', 'Sales']
cities = ['Bangalore', 'Mumbai', 'Delhi', 'Chennai', 'Hyderabad']

for start in range(0, TOTAL_ROWS, CHUNK_SIZE):

    rows = min(CHUNK_SIZE, TOTAL_ROWS - start)

    data = {
        "id": np.arange(start + 1, start + rows + 1),
        "name": [fake.name() for _ in range(rows)],
        "email": [fake.email() for _ in range(rows)],
        "city": np.random.choice(cities, rows),
        "company": [fake.company() for _ in range(rows)],
        "salary": np.random.randint(40000, 200000, rows),
        "age": np.random.randint(21, 65, rows),
        "department": np.random.choice(departments, rows),
        "joining_date": [fake.date_between('-10y', 'today') for _ in range(rows)],
        "active": np.random.choice([True, False], rows)
    }

    df = pd.DataFrame(data)

    df.to_csv(
        OUTPUT_FILE,
        mode='a',
        index=False,
        header=(start == 0)
    )

    print(f"Written rows {start} to {start + rows}")

Written rows 0 to 100000
Written rows 100000 to 200000
Written rows 200000 to 300000
Written rows 300000 to 400000
Written rows 400000 to 500000
Written rows 500000 to 600000
Written rows 600000 to 700000
Written rows 700000 to 800000
Written rows 800000 to 900000
Written rows 900000 to 1000000


---

To read a file in pieces, specify a chunksize as a number of rows:

In [63]:
chunker = pd.read_csv('examples/large.csv', chunksize=100000)

FileNotFoundError: [Errno 2] No such file or directory: 'examples/large.csv'

In [ ]:
chunker

The `TextParser` object returned by `read_csv` allows you to iterate over the parts of
the file according to the chunksize. For example, we can iterate over `ex6.csv`, aggregating
the value counts in the `'key'` column like so:

In [ ]:
tot = pd.Series([])

In [ ]:
for piece in chunker:
    tot = tot.add(piece['name'].value_counts(), fill_value=0)

In [ ]:
tot

#### The Code

```python
chunker = pd.read_csv('examples/large.csv', chunksize=100000)

tot = pd.Series([])

for piece in chunker:
    tot = tot.add(piece['name'].value_counts(), fill_value=0)
```

---

#### 1️⃣ What `chunksize` does

```python
chunker = pd.read_csv('examples/large.csv', chunksize=100000)
```

Instead of loading the whole file, pandas returns an **iterator**.

Each iteration returns a **DataFrame with 100,000 rows**.

Conceptually:

```
large.csv (1M rows)

chunk 1 → rows 1–100000
chunk 2 → rows 100001–200000
chunk 3 → rows 200001–300000
...
chunk 10
```

So memory usage stays **very small**.

---

#### 2️⃣ The variable `chunker`

`chunker` is a **TextFileReader object** (iterator).

You can verify:

```python
type(chunker)
```

Output:

```
pandas.io.parsers.readers.TextFileReader
```

When used in a loop:

```python
for piece in chunker:
```

Each `piece` is a **DataFrame chunk**.

Example structure:

| id | name  | age | city      |
| -- | ----- | --- | --------- |
| 1  | Rahul | 34  | Bangalore |
| 2  | Sneha | 29  | Mumbai    |

---

#### 3️⃣ Why create an empty Series

```python
tot = pd.Series([])
```

This will store the **global counts of each name**.

Example goal:

| name  | count  |
| ----- | ------ |
| Rahul | 152340 |
| Sneha | 120112 |
| Amit  | 98721  |

---

#### 4️⃣ What happens inside the loop

For each chunk:

```python
piece['name'].value_counts()
```

Suppose a chunk has:

```
Rahul
Sneha
Rahul
Amit
Rahul
```

Then:

```python
piece['name'].value_counts()
```

returns

```
Rahul    3
Sneha    1
Amit     1
```

This is a **Series**.

---

#### 5️⃣ The key operation: `.add()`

```python
tot = tot.add(piece['name'].value_counts(), fill_value=0)
```

This **adds the counts from this chunk to the global total**.

Example:

##### First chunk

```
Rahul 3
Sneha 1
Amit 1
```

So

```
tot =
Rahul 3
Sneha 1
Amit 1
```

---

##### Second chunk

```
Rahul 2
Amit 4
Priya 1
```

Now we add:

```
tot.add(new_counts, fill_value=0)
```

Result:

```
Rahul 5
Sneha 1
Amit 5
Priya 1
```

---

#### 6️⃣ Why `fill_value=0` is important

Without it:

```
Rahul + NaN = NaN
```

With `fill_value=0`:

```
Rahul + 0 = Rahul
```

So missing values become **0**.

---

#### 7️⃣ Conceptually what this is

This is the **MapReduce pattern** used in big data systems like:

* Hadoop
* Spark
* Dask

##### Map step

Process each chunk:

```
piece['name'].value_counts()
```

##### Reduce step

Aggregate results:

```
tot.add(...)
```

---

#### 8️⃣ Final result

After the loop finishes:

```
tot
```

contains **name frequencies for the entire file**.

Example:

```
Rahul     153245
Sneha     140002
Amit      134221
Priya     118900
```

---

#### 9️⃣ Modern Pandas Improvement

Pandas warns about this:

```pythonm
tot = pd.Series([])
```

Better version:

```python
tot = pd.Series(dtype='int64')
```

---

#### Final Correct Version

```python
chunker = pd.read_csv('examples/large.csv', chunksize=100000)

tot = pd.Series(dtype='int64')

for piece in chunker:
    tot = tot.add(piece['name'].value_counts(), fill_value=0)

print(tot.sort_values(ascending=False))
```

---

#### Why This Pattern Is Powerful

If the file is **100GB**, this still works because:

You only load **100k rows at a time**.

Memory usage stays small.

---


In [ ]:
tot = tot.sort_values(ascending=False)

In [ ]:
tot[:10]

`TextParser` is also equipped with a `get_chunk` method that enables you to read
pieces of an arbitrary size.

---

A small Utility : 

Below is a clean, reusable data-engineering style utility function that does exactly what you asked:

✔ Accepts file path + name as a single parameter
✔ Prints file size on disk
✔ Loads the file into a temporary DataFrame
✔ Uses chunked loading if file > 1 MB
✔ Reports both:

Disk size

DataFrame memory usage

✔ Automatically formats results in MB or GB

This is the kind of utility function data scientists often build for dataset inspection before analysis.

In [ ]:
import os
import time
import pandas as pd


def inspect_csv_file(file_path):

    # ---------------------------------
    # Helper: Convert bytes to MB / GB
    # ---------------------------------
    def format_size(bytes_size):
        if bytes_size >= 1024**3:
            return f"{bytes_size / (1024**3):.2f} GB"
        else:
            return f"{bytes_size / (1024**2):.2f} MB"

    # ---------------------------------
    # File Size on Disk
    # ---------------------------------
    file_size = os.path.getsize(file_path)

    print("\nFILE INFORMATION")
    print("-----------------------------")
    print(f"File Path : {file_path}")
    print(f"File Size (Disk) : {format_size(file_size)}")

    ONE_MB = 1024 * 1024
    total_memory = 0

    # ---------------------------------
    # Start timer
    # ---------------------------------
    start_time = time.time()

    if file_size > ONE_MB:

        print("\nLoading file using CHUNKS...")

        for chunk in pd.read_csv(file_path, chunksize=100000):

            mem = chunk.memory_usage(deep=True).sum()
            total_memory += mem

    else:

        print("\nLoading file normally...")

        df = pd.read_csv(file_path)
        total_memory = df.memory_usage(deep=True).sum()

    # ---------------------------------
    # Stop timer
    # ---------------------------------
    end_time = time.time()
    load_time = end_time - start_time

    # ---------------------------------
    # Results
    # ---------------------------------
    print("\nDATAFRAME MEMORY USAGE")
    print("-----------------------------")
    print(f"Estimated DataFrame Memory Usage : {format_size(total_memory)}")

    print("\nPERFORMANCE")
    print("-----------------------------")
    print(f"Time Taken to Load with Pandas : {load_time:.2f} seconds")

In [ ]:
inspect_csv_file("examples/large.csv")

-----

### Writing Data to Text Format

Data can also be exported to a delimited format. Let’s consider one of the CSV files
read before:

In [ ]:
data = pd.read_csv('examples/ex5.csv')

In [ ]:
data

Using DataFrame’s `to_csv` method, we can write the data out to a comma-separated
file:

In [ ]:
data.to_csv('examples/out.csv')

In [ ]:
!type examples\out.csv

Other delimiters can be used, of course (writing to `sys.stdout` so it prints the text
result to the console):

In [ ]:
import sys

In [ ]:
data.to_csv(sys.stdout, sep='|')

Missing values appear as empty strings in the output. You might want to denote them
by some other sentinel value:

In [ ]:
data.to_csv(sys.stdout, na_rep='NULL')

With no other options specified, both the row and column labels are written. Both of
these can be disabled:

In [ ]:
data.to_csv(sys.stdout, index=False, header=False)

You can also write only a subset of the columns, and in an order of your choosing:

In [ ]:
data.to_csv(sys.stdout, index=False, columns=['a', 'b', 'c'])

---

Series also has a `to_csv` method:

In [ ]:
dates = pd.date_range('1/1/2000', periods=7)

In [ ]:
ts = pd.Series(np.arange(7), index=dates)

In [ ]:
ts.to_csv('examples/tseries.csv')

In [ ]:
!type examples\tseries.csv

### Working with Delimited Formats

It’s possible to load most forms of tabular data from disk using functions like `pandas.read_table.` In some cases, however, some manual processing may be necessary.
It’s not uncommon to receive a file with one or more malformed lines that trip up
read_table. To illustrate the basic tools, consider a small CSV file:

In [ ]:
%%writefile examples/ex7.csv
"a","b","c"
"1","2","3"
"1","2","3"

For any file with a single-character delimiter, you can use Python’s built-in `csv` module.
To use it, pass any open file or file-like object to `csv.reader`:

In [ ]:
import csv
f = open('examples/ex7.csv')
reader = csv.reader(f)

Iterating through the reader like a file yields tuples of values with any quote characters
removed:

In [ ]:
for line in reader:
    print(line)

From there, it’s up to you to do the wrangling necessary to put the data in the form
that you need it. Let’s take this step by step. First, we read the file into a list of lines:

In [ ]:
with open('examples/ex7.csv') as f:
    lines = list(csv.reader(f))

Then, we split the lines into the header line and the data lines:

In [ ]:
header, values = lines[0], lines[1:]

Then we can create a dictionary of data columns using a dictionary comprehension
and the expression `zip(*values)`, which transposes rows to columns

In [ ]:
data_dict = {h: v for h, v in zip(header, zip(*values))}

In [ ]:
data_dict

CSV files come in many different flavors. To define a new format with a different
delimiter, string quoting convention, or line terminator, we define a simple subclass
of `csv.Dialect`:

In [ ]:
class my_dialect(csv.Dialect):
    lineterminator = '\n'
    delimiter = ';'
    quotechar = '"'
    quoting = csv.QUOTE_MINIMAL


In [ ]:
f = open('examples/ex7.csv')
reader = csv.reader(f, dialect=my_dialect)

We can also give individual CSV dialect parameters as keywords to `csv.reader`
without having to define a subclass:


In [ ]:
reader = csv.reader(f, delimiter='|')

The possible options (attributes of csv.Dialect) and what they do can be found in
Table 6-3.

In [ ]:
columns = ["Argument", "Description"]

rows = [
["delimiter","One-character string to separate fields; defaults to ','"],
["lineterminator","Line terminator for writing; defaults to '\\r\\n'. Reader ignores this and recognizes cross-platform line terminators"],
["quotechar","Quote character for fields with special characters (like a delimiter); default is '\"'"],
["quoting","Quoting convention. Options include csv.QUOTE_ALL (quote all fields), csv.QUOTE_MINIMAL (only fields with special characters like the delimiter), csv.QUOTE_NONNUMERIC, and csv.QUOTE_NONE (no quoting). Defaults to QUOTE_MINIMAL"],
["skipinitialspace","Ignore whitespace after each delimiter; default is False"],
["doublequote","How to handle quoting character inside a field; if True, it is doubled"],
["escapechar","String to escape the delimiter if quoting is set to csv.QUOTE_NONE; disabled by default"]
]

render_book_table(
    "Table 6-3. CSV Dialect Options",
    columns,
    rows
)

To write delimited files manually, you can use `csv.writer`. It accepts an open, writable
file object and the same dialect and format options as csv.reader:


In [ ]:
with open('examples/mydata.csv', 'w') as f:
    writer = csv.writer(f, dialect=my_dialect)
    writer.writerow(('one', 'two', 'three'))
    writer.writerow(('1', '2', '3'))
    writer.writerow(('4', '5', '6'))
    writer.writerow(('7', '8', '9'))

In [ ]:
!type examples\mydata.csv

### JSON Data

JSON (short for JavaScript Object Notation) has become one of the standard formats
for sending data by HTTP request between web browsers and other applications. It is
a much more free-form data format than a tabular text form like CSV. Here is an
example:

In [ ]:
obj = """
{"name": "Wes",
"places_lived": ["United States", "Spain", "Germany"],
"pet": null,
"siblings": [{"name": "Scott", "age": 30, "pets": ["Zeus", "Zuko"]},
{"name": "Katie", "age": 38,
"pets": ["Sixes", "Stache", "Cisco"]}]
}
"""

JSON is very nearly valid Python code with the exception of its null value null and
some other nuances (such as disallowing trailing commas at the end of lists). The
basic types are objects (dicts), arrays (lists), strings, numbers, booleans, and nulls. All
of the keys in an object must be strings. There are several Python libraries for reading and writing JSON data. I’ll use `json` here, as it is built into the Python standard
library. To convert a JSON string to Python form, use `json.loads`:

In [ ]:
import json

In [119]:
result = json.loads(obj)

In [120]:
result

{'name': 'Wes',
 'places_lived': ['United States', 'Spain', 'Germany'],
 'pet': None,
 'siblings': [{'name': 'Scott', 'age': 30, 'pets': ['Zeus', 'Zuko']},
  {'name': 'Katie', 'age': 38, 'pets': ['Sixes', 'Stache', 'Cisco']}]}

In [122]:
type(result)

dict

`json.dumps`, on the other hand, converts a Python object back to JSON:


In [124]:
asjson = json.dumps(result)

In [125]:
asjson

'{"name": "Wes", "places_lived": ["United States", "Spain", "Germany"], "pet": null, "siblings": [{"name": "Scott", "age": 30, "pets": ["Zeus", "Zuko"]}, {"name": "Katie", "age": 38, "pets": ["Sixes", "Stache", "Cisco"]}]}'

How you convert a JSON object or list of objects to a DataFrame or some other data
structure for analysis will be up to you. Conveniently, you can pass a list of dicts
(which were previously JSON objects) to the DataFrame constructor and select a subset
of the data fields:

In [126]:
siblings = pd.DataFrame(result['siblings'], columns=['name', 'age'])

In [127]:
siblings

,name,age
0,Scott,30
1,Katie,38


The `pandas.read_json` can automatically convert JSON datasets in specific arrangements
into a Series or DataFrame. For example:

In [128]:
%%writefile examples/example.json
[{"a": 1, "b": 2, "c": 3},
{"a": 4, "b": 5, "c": 6},
{"a": 7, "b": 8, "c": 9}]

Writing examples/example.json


The default options for `pandas.read_json` assume that each object in the JSON array
is a row in the table:

In [130]:
data = pd.read_json('examples/example.json')

In [131]:
data

,a,b,c
0,1,2,3
1,4,5,6
2,7,8,9


For an extended example of reading and manipulating JSON data (including nested
records), see the USDA Food Database example in Chapter 7.

If you need to export data from pandas to JSON, one way is to use the `to_json` methods
on Series and DataFrame:

In [135]:
print(data.to_json())

{"a":{"0":1,"1":4,"2":7},"b":{"0":2,"1":5,"2":8},"c":{"0":3,"1":6,"2":9}}


In [136]:
print(data.to_json(orient='records'))

[{"a":1,"b":2,"c":3},{"a":4,"b":5,"c":6},{"a":7,"b":8,"c":9}]


### XML and HTML: Web Scraping

Python has many libraries for reading and writing data in the ubiquitous HTML and
XML formats. Examples include lxml, Beautiful Soup, and html5lib. While lxml is
comparatively much faster in general, the other libraries can better handle malformed
HTML or XML files.

pandas has a built-in function, `read_html`, which uses libraries like lxml and Beautiful
Soup to automatically parse tables out of HTML files as DataFrame objects. To
show how this works, I downloaded an HTML file (used in the pandas documentation)
from the United States FDIC government agency showing bank failures.
(donload the HTML file from https://www.fdic.gov/bank/individual/failed/banklist.html to your examples folder)

First, you must install some additional libraries used by `read_html`:



First Isntall lxml package in same Python environment which is used by Jupyter.

`conda install lxml`

The `pandas.read_html` function has a number of options, but by default it searches
for and attempts to parse all tabular data contained within `<table>` tags. The result is
a list of DataFrame objects:

In [157]:
tables = pd.read_html('examples/failed.gov.html')

In [158]:
len(tables)

1

In [159]:
failures = tables[0]

In [160]:
failures

,Bank Name,City,State,Cert,Acquiring Institution,Closing Date,Fund Sort ascending
0,Metropolitan Capital Bank & Trust,Chicago,Illinois,57488,First Independence Bank,"January 30, 2026",10550
1,The Santa Anna National Bank,Santa Anna,Texas,5520,Coleman County State Bank,"June 27, 2025",10549
2,Pulaski Savings Bank,Chicago,Illinois,28611,Millennium Bank,"January 17, 2025",10548
3,The First National Bank of Lindsay,Lindsay,Oklahoma,4134,First Bank & Trust Co.,"October 18, 2024",10547
4,Republic First Bank dba Republic Bank,Philadelphia,Pennsylvania,27332,"Fulton Bank, National Association","April 26, 2024",10546
...,...,...,...,...,...,...,...
568,Sinclair National Bank,Gravette,Arkansas,34248,Delta Trust & Bank,"September 7, 2001",4649
569,Malta National Bank,Malta,Ohio,6629,North Valley Bank,"May 3, 2001",4648
570,First Alliance Bank & Trust Co.,Manchester,New Hampshire,34264,Southern New Hampshire Bank & Trust,"February 2, 2001",4647
571,The National State Bank of Metropolis,Metropolis,Illinois,3815,Banterra Bank of Marion,"December 14, 2000",4646


As you will learn in later chapters, from here we could proceed to do some data
cleaning and analysis, like computing the number of bank failures by year:

In [161]:
close_timestamps = pd.to_datetime(failures['Closing Date'])

In [162]:
close_timestamps.dt.year.value_counts()

Closing Date
2010    157
2009    140
2011     92
2012     51
2008     25
       ... 
2007      3
2000      2
2025      2
2024      2
2026      1
Name: count, Length: 22, dtype: int64

### Parsing XML with lxml.objectify

XML (eXtensible Markup Language) is another common structured data format supporting
hierarchical, nested data with metadata. The book you are currently reading
was actually created from a series of large XML documents.

Earlier, I showed the pandas.read_html function, which uses either lxml or Beautiful
Soup under the hood to parse data from HTML. XML and HTML are structurally
similar, but XML is more general. Here, I will show an example of how to use lxml to
parse data from a more general XML format.

The New York Metropolitan Transportation Authority (MTA) publishes a number of
data series about its bus and train services. Here we’ll look at the performance data,
which is contained in a set of XML files. 

https://data.ny.gov/api/views/ks33-g5ze/rows.xml?accessType=DOWNLOAD

##### XML Structure (Simplified)

Your XML structure looks like this (simplified):

```xml
<response>
    <row>
        <row>
            <month>2025-01-01T00:00:00</month>
            <division>A DIVISION</division>
            <line>1</line>
            <day_type>1</day_type>
            <num_on_time_trips>8510</num_on_time_trips>
            <num_sched_trips>10164</num_sched_trips>
            <terminal_on_time_performance>0.83726879</terminal_on_time_performance>
        </row>
        ...
    </row>
</response>
```

So the records are inside:

```
root.row.row
```

---

##### Python Code to Read Elements from the XML

Below is **Python code using `lxml.objectify`**, written **very similar to the style you showed**.



In [190]:
from lxml import objectify

path = 'examples/mta_perf_rows.xml'

parsed = objectify.parse(open(path))
root = parsed.getroot()

data = []

skip_fields = ['_id', '_uuid', '_position', '_address']

for elt in root.row.row:

    el_data = {}

    for child in elt.getchildren():

        if child.tag in skip_fields:
            continue

        el_data[child.tag] = child.pyval

    data.append(el_data)

print(data[:5])


[{'month': '2025-01-01T00:00:00', 'division': 'A DIVISION', 'line': 1, 'day_type': 1, 'num_on_time_trips': 8510, 'num_sched_trips': 10164, 'terminal_on_time_performance': 0.8372687918142464}, {'month': '2025-01-01T00:00:00', 'division': 'A DIVISION', 'line': 1, 'day_type': 2, 'num_on_time_trips': 2335, 'num_sched_trips': 2847, 'terminal_on_time_performance': 0.8201615735862311}, {'month': '2025-01-01T00:00:00', 'division': 'A DIVISION', 'line': 2, 'day_type': 1, 'num_on_time_trips': 5065, 'num_sched_trips': 7100, 'terminal_on_time_performance': 0.7133802816901408}, {'month': '2025-01-01T00:00:00', 'division': 'A DIVISION', 'line': 2, 'day_type': 2, 'num_on_time_trips': 1358, 'num_sched_trips': 2140, 'terminal_on_time_performance': 0.6345794392523364}, {'month': '2025-01-01T00:00:00', 'division': 'A DIVISION', 'line': 3, 'day_type': 1, 'num_on_time_trips': 5396, 'num_sched_trips': 6562, 'terminal_on_time_performance': 0.8223102712587625}]


Optional: Convert Directly to Pandas DataFrame

Since you are doing data science analysis, you will usually convert this list to a dataframe.

In [191]:
import pandas as pd

df = pd.DataFrame(data)

df.head()

,month,division,line,day_type,num_on_time_trips,num_sched_trips,terminal_on_time_performance
0,2025-01-01T00:00:00,A DIVISION,1,1,8510,10164,0.837269
1,2025-01-01T00:00:00,A DIVISION,1,2,2335,2847,0.820162
2,2025-01-01T00:00:00,A DIVISION,2,1,5065,7100,0.713380
3,2025-01-01T00:00:00,A DIVISION,2,2,1358,2140,0.634579
4,2025-01-01T00:00:00,A DIVISION,3,1,5396,6562,0.822310



##### Small Improvement (More Robust Code)

Sometimes XML has attributes. This version ignores them safely.


In [193]:
for elt in root.row.row:
    el_data = {}

    for child in elt.iterchildren():
        el_data[child.tag] = child.pyval

    data.append(el_data)

df = pd.DataFrame(data)
df.head()

,month,division,line,day_type,num_on_time_trips,num_sched_trips,terminal_on_time_performance
0,2025-01-01T00:00:00,A DIVISION,1,1,8510,10164,0.837269
1,2025-01-01T00:00:00,A DIVISION,1,2,2335,2847,0.820162
2,2025-01-01T00:00:00,A DIVISION,2,1,5065,7100,0.713380
3,2025-01-01T00:00:00,A DIVISION,2,2,1358,2140,0.634579
4,2025-01-01T00:00:00,A DIVISION,3,1,5396,6562,0.822310


---

XML data can get much more complicated than this example. Each tag can have
metadata, too. Consider an HTML link tag, which is also valid XML:

In [180]:
from io import StringIO

In [181]:
tag = '<a href="http://www.google.com">Google</a>'

In [182]:
root = objectify.parse(StringIO(tag)).getroot()

You can now access any of the fields (like `href`) in the tag or the link text:

In [184]:
root.get('href')

'http://www.google.com'

In [188]:
root.text

'Google'

## 6.2 Binary Data Formats

One of the easiest ways to store data (also known as serialization) efficiently in binary
format is using Python’s built-in `pickle` serialization. pandas objects all have a
`to_pickle` method that writes the data to disk in pickle format:

In [194]:
frame = pd.read_csv('examples/ex1.csv')

In [195]:
frame

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


In [196]:
frame.to_pickle('examples/frame_pickle')

You can read any “pickled” object stored in a file by using the built-in `pickle` directly,
or even more conveniently using `pandas.read_pickle`:

In [198]:
pd.read_pickle('examples/frame_pickle')

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


## 🦂 
> pickle is only recommended as a short-term storage format. The
problem is that it is hard to guarantee that the format will be stable
over time; an object pickled today may not unpickle with a later
version of a library. We have tried to maintain backward compatibility
when possible, but at some point in the future it may be necessary
to “break” the pickle format.

pandas has built-in support for two more binary data formats: HDF5 and Message‐
Pack. I will give some HDF5 examples in the next section, but I encourage you to
explore different file formats to see how fast they are and how well they work for your
analysis. Some other storage formats for pandas or NumPy data include:

#### bcolz
> A compressable column-oriented binary format based on the Blosc compression
library.
#### Feather
> A cross-language column-oriented file format I designed with the R programming
community’s Hadley Wickham. Feather uses the Apache Arrow columnar
memory format.

### Using HDF5 Format

HDF5 is a well-regarded file format intended for storing large quantities of scientific
array data. It is available as a C library, and it has interfaces available in many other
languages, including Java, Julia, MATLAB, and Python. The “HDF” in HDF5 stands
for hierarchical data format. Each HDF5 file can store multiple datasets and supporting
metadata. Compared with simpler formats, HDF5 supports on-the-fly compression
with a variety of compression modes, enabling data with repeated patterns to be
stored more efficiently. HDF5 can be a good choice for working with very large datasets
that don’t fit into memory, as you can efficiently read and write small sections of
much larger arrays.

While it’s possible to directly access HDF5 files using either the PyTables or h5py
libraries, pandas provides a high-level interface that simplifies storing Series and
DataFrame object. The `HDFStore` class works like a dict and handles the low-level
details:

In [202]:
frame = pd.DataFrame({'a': np.random.randn(100)})

In [203]:
frame

,a
0,0.569398
1,-0.712453
2,0.359564
3,0.124748
4,0.062806
...,...
95,0.570122
96,-1.414933
97,-0.107870
98,0.789606


In [208]:
store = pd.HDFStore('examples/mydata.h5')

In [209]:
store['obj1'] = frame

In [210]:
store['obj1_col'] = frame['a']

In [211]:
store

<class 'pandas.io.pytables.HDFStore'>
File path: examples/mydata.h5

In [212]:
store['obj1']

,a
0,0.569398
1,-0.712453
2,0.359564
3,0.124748
4,0.062806
...,...
95,0.570122
96,-1.414933
97,-0.107870
98,0.789606


HDFStore supports two storage schemas, `'fixed'` and `'table'`. The latter is generally
slower, but it supports query operations using a special syntax:

In [214]:
store.put('obj2', frame, format='table')

In [215]:
store.select('obj2', where=['index >= 10 and index <= 15'])

,a
10,-1.084569
11,-0.443398
12,-0.353793
13,0.419334
14,0.310127
15,0.272252


In [216]:
store.close()

The put is an explicit version of the `store['obj2']` = frame method but allows us to
set other options like the storage format.

The `pandas.read_hdf` function gives you a shortcut to these tools:

In [219]:
frame.to_hdf('mydata.h5', 'obj3', format='table')

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_32472\1098501470.py:1: FutureWarning: Starting with pandas version 3.0 all arguments of to_hdf except for the argument 'path_or_buf' will be keyword-only.
  frame.to_hdf('mydata.h5', 'obj3', format='table')


In [220]:
pd.read_hdf('mydata.h5', 'obj3', where=['index < 5'])

ValueError: The file 'mydata.h5' is already opened, but not in read-only mode (as requested).

----

> HDF5 is not a database. It is best suited for write-once, read-many
datasets. While data can be added to a file at any time, if multiple
writers do so simultaneously, the file can become corrupted.

If you work with large quantities of data locally, I would encourage you to explore
PyTables and h5py to see how they can suit your needs. Since many data analysis
problems are I/O-bound (rather than CPU-bound), using a tool like HDF5 can massively
accelerate your applications.

### Reading Microsoft Excel Files

pandas also supports reading tabular data stored in Excel 2003 (and higher) files
using either the `ExcelFile` class or `pandas.read_excel` function. Internally these
tools use the add-on packages `xlrd` and o`penpyxl` to read XLS and XLSX files, respectively.
You may need to install these manually with pip or conda.

To use ExcelFile, create an instance by passing a path to an `xls` or `xlsx` file:

In [224]:
xlsx = pd.ExcelFile('examples/ex2.xlsx')

Data stored in a sheet can then be read into DataFrame with parse:

In [226]:
pd.read_excel(xlsx, 'Sheet1')

,a,b,c,d
0,1,2,3,4
1,5,6,7,8
2,9,10,11,12


If you are reading multiple sheets in a file, then it is faster to create the ExcelFile,
but you can also simply pass the filename to `pandas.read_excel`:

In [230]:
frame = pd.read_excel('examples/ex2.xlsx', 'Sheet1')

In [231]:
frame

,a,b,c,d
0,1,2,3,4
1,5,6,7,8
2,9,10,11,12


To write pandas data to Excel format, you must first create an `ExcelWriter`, then
write data to it using pandas objects’ to_excel method:

In [232]:
writer = pd.ExcelWriter('examples/ex3.xlsx')

In [233]:
frame.to_excel(writer, 'Sheet1')

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_32472\884453241.py:1: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  frame.to_excel(writer, 'Sheet1')


In [235]:
writer.close()

You can also pass a file path to `to_excel` and avoid the `ExcelWriter`:

In [237]:
frame.to_excel('examples/ex4.xlsx')

## 6.3 Interacting with Web APIs

Many websites have public APIs providing data feeds via JSON or some other format.
There are a number of ways to access these APIs from Python; one easy-to-use
method that I recommend is the `requests` package.

To find the last 30 GitHub issues for pandas on GitHub, we can make a `GET` HTTP
request using the add-on `requests` library:

In [21]:
import requests

In [22]:
url = 'https://api.github.com/repos/pandas-dev/pandas/issues'

In [23]:
resp = requests.get(url)

In [24]:
resp

<Response [200]>

In [25]:
data = resp.json()

In [26]:
data

[{'url': 'https://api.github.com/repos/pandas-dev/pandas/issues/64701',
  'repository_url': 'https://api.github.com/repos/pandas-dev/pandas',
  'labels_url': 'https://api.github.com/repos/pandas-dev/pandas/issues/64701/labels{/name}',
  'comments_url': 'https://api.github.com/repos/pandas-dev/pandas/issues/64701/comments',
  'events_url': 'https://api.github.com/repos/pandas-dev/pandas/issues/64701/events',
  'html_url': 'https://github.com/pandas-dev/pandas/pull/64701',
  'id': 4098683910,
  'node_id': 'PR_kwDOAA0YD87Lt0ba',
  'number': 64701,
  'title': 'DEPR: Series.dt.freq returning a string instead of DateOffset',
  'user': {'login': 'jbrockmendel',
   'id': 8078968,
   'node_id': 'MDQ6VXNlcjgwNzg5Njg=',
   'avatar_url': 'https://avatars.githubusercontent.com/u/8078968?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/jbrockmendel',
   'html_url': 'https://github.com/jbrockmendel',
   'followers_url': 'https://api.github.com/users/jbrockmendel/followers',
   'fol

In [245]:
    data[0]['title']

'CLN: Remove redundant .format() on f-strings in _IntArrayFormatter'

Each element in data is a dictionary containing all of the data found on a GitHub
issue page (except for the comments). We can pass `data` directly to DataFrame and
extract fields of interest:

In [247]:
issues = pd.DataFrame(data, columns=['number', 'title','labels', 'state'])

In [248]:
issues

,number,title,labels,state
0,64474,CLN: Remove redundant .format() on f-strings i...,[],open
1,64473,CLN: Fix typo :meth and :class:CategoricalIndex,[],open
2,64472,CLN: Hide Categorical.reshape/swapaxes/transpo...,[],open
3,64471,DOC: Add asi8 and unit properties to API refer...,[],open
4,64470,DOC: Fix ES01 for pandas.DatetimeTZDtype,[],open
...,...,...,...,...
25,64432,REF: use fast_float instead of bespoke str-to-...,[],open
26,64429,PERF: Avoid intermediate `ndarray[object]` arr...,"[{'id': 8935311, 'node_id': 'MDU6TGFiZWw4OTM1M...",open
27,64428,BUG: `DataFrame.div()` ignores `axis`,"[{'id': 76811, 'node_id': 'MDU6TGFiZWw3NjgxMQ=...",open
28,64425,BUG: fix non-unitary anchored resample edge al...,[],open


With a bit of elbow grease, you can create some higher-level interfaces to common
web APIs that return DataFrame objects for easy analysis.

## 6.4 Interacting with Databases

In a business setting, most data may not be stored in text or Excel files. SQL-based
relational databases (such as SQL Server, PostgreSQL, and MySQL) are in wide use,
and many alternative databases have become quite popular. The choice of database is
usually dependent on the performance, data integrity, and scalability needs of an
application.

Loading data from SQL into a DataFrame is fairly straightforward, and pandas has
some functions to simplify the process. As an example, I’ll create a SQLite database
using Python’s built-in `sqlite3` driver:

In [16]:
import sqlite3

In [17]:
query = """CREATE TABLE test (a VARCHAR(20), b VARCHAR(20),c REAL, d INTEGER );"""

In [18]:
con = sqlite3.connect('otherfiles/mydata.sqlite')

In [5]:
con.execute(query)

In [6]:
con.commit()

In [7]:
data = [('Atlanta', 'Georgia', 1.25, 6), ('Tallahassee', 'Florida', 2.6, 3),('Sacramento', 'California', 1.7, 5)]

In [8]:
stmt = "INSERT INTO test VALUES(?, ?, ?, ?)"

In [9]:
con.executemany(stmt, data)

In [10]:
con.commit()

Most Python SQL drivers (PyODBC, psycopg2, MySQLdb, pymssql, etc.) return a list
of tuples when selecting data from a table:

In [11]:
cursor = con.execute('select * from test')

In [12]:
rows = cursor.fetchall()

In [13]:
rows

[('Atlanta', 'Georgia', 1.25, 6),
 ('Tallahassee', 'Florida', 2.6, 3),
 ('Sacramento', 'California', 1.7, 5)]

You can pass the list of tuples to the DataFrame constructor, but you also need the
column names, contained in the cursor’s `description` attribute:

In [14]:
cursor.description

(('a', None, None, None, None, None, None),
 ('b', None, None, None, None, None, None),
 ('c', None, None, None, None, None, None),
 ('d', None, None, None, None, None, None))

In [15]:
pd.DataFrame(rows, columns=[x[0] for x in cursor.description])

NameError: name 'pd' is not defined

---

This is quite a bit of munging that you’d rather not repeat each time you query the
database. The SQLAlchemy project is a popular Python SQL toolkit that abstracts
away many of the common differences between SQL databases. pandas has a
`read_sql` function that enables you to read data easily from a general SQLAlchemy
connection. Here, we’ll connect to the same SQLite database with SQLAlchemy and
read data from the table created before:

In [265]:
import sqlalchemy as sqla

In [266]:
db = sqla.create_engine('sqlite:///mydata.sqlite')

In [267]:
pd.read_sql('select * from test', db)

,a,b,c,d
0,Atlanta,Georgia,1.25,6
1,Tallahassee,Florida,2.60,3
2,Sacramento,California,1.70,5


## 6.5 Conclusion

Getting access to data is frequently the first step in the data analysis process. We have
looked at a number of useful tools in this chapter that should help you get started. In
the upcoming chapters we will dig deeper into data wrangling, data visualization,
time series analysis, and other topics.